In [ ]:
training_ratio = 0.8
test_ratio = 0.2
validation_ratio = 1 - training_ratio - test_ratio

# Federated Learning parameters
CLIENTS_COUNT = 100
DATASET = "MNIST"
IID = False
EPOCHS = 34

# SpyShield Parameters
CLIENTS_PER_MODEL = 10
BIAS = 9
MODELS_COUNT = (CLIENTS_COUNT - 1) // (CLIENTS_PER_MODEL - 1)

# poisoning scenario
SCENARIO = 3
# ratio of poisoned clients
ratio = 0.4

# set the device to either "cuda" when using the gpu or "cpu" when using the cpu
device = "cuda"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# used in printing to a file
import sys

# GMM
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split

# used to deep copy the global models to local models
import copy

# tracks progress of each loops
from tqdm import tqdm

# pytorch
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, random_split, Subset, ConcatDataset
from torchvision import datasets, transforms

In [ ]:
def getDataset(dataset):
    if(dataset == "MNIST"):
    
        transform = transforms.Compose([
            transforms.ToTensor(), 
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        
        training_dataset = datasets.MNIST('../data', train=True, download=True, transform=transform)
    
    elif(dataset == "FashionMNIST"):
         
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.2860,), (0.3527,))
        ])
    
        training_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
    return training_dataset

In [ ]:
def sample(dataset, iid = False, clients = 30, classes = 2, training_ratio = 0.8, test_ratio = 0.2):
 
    if(not iid):
        # Initialize dictionary for indices
        indicesByClass = {i: [] for i in range(10)}  
        for idx, (_, label) in enumerate(dataset):
            indicesByClass[label].append(idx)  # Append the index to the corresponding class
    
        # List of subsets for each class 
        datasetsByClass = [Subset(dataset, indices) for indices in indicesByClass.values()]

        rng = np.random.default_rng()
        shards = rng.integers(classes,int(1000/clients), size=clients)
        shards = (shards * 1000/sum(shards)).astype(int)

        datasetsByClient = []
        for i, shard in enumerate(shards):
            datasets = []
            shardPerClass = int(shard/classes)
            pairs = []
            for j in range(len(datasetsByClass)):
                pairs.append((j,len(datasetsByClass[j])))
            rng.shuffle(pairs)
            for j in range(classes):
                pair = max(pairs, key=lambda x: x[1])
                index = pair[0]
                pairs.remove(pair)
                length = min(shardPerClass * int(len(dataset)/1000), len(datasetsByClass[index]))
                datasets.append(Subset(datasetsByClass[index], np.arange(length)))
                datasetsByClass[index] = Subset(datasetsByClass[index], range(length, len(datasetsByClass[index])))
            datasetsByClient.append(ConcatDataset(datasets))

    else:
        datasetsByClient = random_split(dataset, np.ones(clients, dtype = int)*(len(dataset)//clients))
    
    for i in range(len(datasetsByClient)):
        training_length = int(len(datasetsByClient[i]) * training_ratio)
        test_length = int(len(datasetsByClient[i]) * test_ratio)
        validation_length = len(datasetsByClient[i]) - training_length - test_length
        
        split = [training_length, test_length, validation_length]
        datasetsByClient[i] = tuple(random_split(datasetsByClient[i], split))
    
    return datasetsByClient

In [ ]:
# convolutional neural network
class MNIST(nn.Module):
    def __init__(self):
        super(MNIST, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, x.shape[1]*x.shape[2]*x.shape[3])
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)
    

In [ ]:
class FashionMNIST(nn.Module):
    def __init__(self):
        super(FashionMNIST, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2))
        self.fc = nn.Linear(7*7*32, 10)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

In [ ]:
def Model():
    if(DATASET == "MNIST"):
        return MNIST()
    elif(DATASET == "FashionMNIST"):
        return FashionMNIST()

In [ ]:
def train(model, global_round, client, dataset, malicious, scenario = 1, epochs = 10, display = False):
    data_loader = DataLoader(dataset, batch_size = 10, shuffle=not malicious)
    criterion = nn.NLLLoss().to('device')
    # Set mode to train model
    model.to('device')
    model.train()
    epoch_loss = []

    # Set optimizer for the local updates
    optimizer = torch.optim.SGD(model.parameters(), lr = 0.001 ,momentum=0.5)
    
    for epoch in range(epochs):
        batch_loss = []
        
        for batch_idx, (images, labels) in enumerate(data_loader):

            # poisoning
            if(malicious):
                if(scenario == 1):
                    for i in range(len(labels)):
                        labels[i] = (labels[i] + rng.integers(1,10)) % 10
                elif(scenario == 2):
                    for i in range(len(labels)):
                        labels[i] = (labels[i] + 5)% 10
                elif(scenario == 3):
                    for i in range(len(labels)):
                        if (labels[i] == 0):
                            labels[i] = 9
                        
            images, labels = images.to('device'), labels.to('device')

            model.zero_grad()
            log_probs = model(images)
            loss = criterion(log_probs, labels)
            loss.backward()
            optimizer.step()
            if(display):
                print('| Global Round : {} | Client : {} | Local Epoch : {} | [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                    global_round + 1, client + 1, epoch + 1, batch_idx * len(images),
                    len(data_loader.dataset),
                    100. * batch_idx / len(data_loader), loss.item())
                )
            
            batch_loss.append(loss.item())
        epoch_loss.append(sum(batch_loss)/len(batch_loss))

    return model.state_dict(), sum(epoch_loss) / len(epoch_loss)

In [ ]:
def test(model, dataset, malicious, scenario = 1):
    criterion = nn.NLLLoss().to('device')
    
    data_loader = DataLoader(dataset, batch_size = 8, shuffle=False)
    # set the model to evaluation mode.
    model.to('device')
    model.eval()
    loss, total, correct = 0.0, 0.0, 0.0
    
    for batch_idx, (images, labels) in enumerate(data_loader):
        
        # poisoning
        if(malicious):
                if(scenario == 2):
                    for i in range(len(labels)):
                        labels[i] = (labels[i] + 5)% 10
                elif(scenario == 3):
                    for i in range(len(labels)):
                        if (labels[i] == 0):
                            labels[i] = 9
                        
        images, labels = images.to('device'), labels.to('device')

        # Inference
        outputs = model(images)
        batch_loss = criterion(outputs, labels)
        loss += batch_loss.item()

        # Prediction
        _, pred_labels = torch.max(outputs, 1)
        pred_labels = pred_labels.view(-1)
        correct += torch.sum(torch.eq(pred_labels, labels)).item()
        total += len(labels)

    accuracy = correct/total
    return accuracy, loss

In [ ]:
def average(w, bias = 1):
    avg = copy.deepcopy(w[0])
    for key in avg.keys():
        for i in range(1,bias):
            avg[key] += w[0][key]
        for i in range(1, len(w)):
            avg[key] += w[i][key]
        avg[key] = torch.div(avg[key], len(w) + bias - 1)
    return avg

In [ ]:
def group(CLIENTS_COUNT, CLIENTS_PER_MODEL):
    MODELS_COUNT = (CLIENTS_COUNT - 1) // (CLIENTS_PER_MODEL - 1)
    groups  = np.empty([CLIENTS_COUNT, MODELS_COUNT], dtype = object)
    groupings  = np.empty([CLIENTS_COUNT, MODELS_COUNT], dtype = object)
    for i in range(CLIENTS_COUNT):
        remaining = np.delete(np.arange(CLIENTS_COUNT), i)
        for j in range(MODELS_COUNT):
                group = np.concatenate(([i],rng.choice(remaining, size= CLIENTS_PER_MODEL - 1, replace=False)))
                groups[i][j] = group
                remaining = np.delete(remaining, np.where(np.isin(remaining, group[1:])))
            
    while(True):
        frequency = np.zeros(CLIENTS_COUNT, dtype=int)
        for i in range(CLIENTS_COUNT):
            for j in range(MODELS_COUNT):
                
                pairs = [(i, frequency[i]) for i in range(len(frequency))]
                filtered = []
                for pair in pairs:
                    if pair[0] not in groups[i][j]:
                        filtered.append(pair)
                    
                rng.shuffle(filtered)
                assignee = min(filtered, key=lambda x: x[1])[0]
                groupings[i][j] = (groups[i][j], assignee)
                # update the frequcny 
                frequency[assignee] += 1;
        if(np.all(frequency == MODELS_COUNT)):
            break
    return groupings

In [ ]:
def toString(s):
    return np.array2string(np.array(s), precision=2, separator=',')

In [ ]:
def isMalicious(client):
    return client in dictionary

In [ ]:
def isReliable(client):
    return not (client in unreliable)

In [ ]:
def isPoisoned(client):
    return client in poisoned

In [ ]:
def isBimodal(data):
    data = np.array(data)
    
    # Test models with 1 and 2 components
    n_components = range(1, 3) 
    # intialize GMMs with 1 and 2 components and fit to the data
    models = [GaussianMixture(n, init_params='k-means++', n_init=10, covariance_type='full').fit(data.reshape(-1, 1)) for n in n_components]

    # Compute Bayesian Information Criterion (BIC) for each model
    bics = [model.bic(data.reshape(-1, 1)) for model in models]
    
    #if the model with the least BIC is the second model in the zero indexed list of models return true 
    return (bics[0]> bics[1])

In [ ]:
def plot(data):
    plt.hist(data)
    plt.show()

In [ ]:
def findBoundary(data):
    data = np.array(data)
    gmm = GaussianMixture(n_components=2, init_params='k-means++', n_init=10, covariance_type='full', random_state=42)
    gmm.fit(data.reshape(-1, 1))
    x = np.linspace(min(data), max(data), 1000).reshape(-1, 1)

    # Compute probability densities for both Gaussian components
    probabilities = gmm.predict_proba(x)
    
    # intialize an array of the difference between the probaility that a datapoint belongs to the first cluster and to the second cluster
    difference = np.abs(probabilities[:, 0] - probabilities[:, 1])

    #find the point with the smallest difference
    threshold = x[np.argmin(difference)][0]

    return threshold

In [ ]:
def observe():
    for i in range(CLIENTS_COUNT):
        for j in range(MODELS_COUNT):
            s = "["
            for k in range(CLIENTS_PER_MODEL):
                client = groupings[i][j][0][k]
                if(isMalicious(groupings[i][j][0][k])):
                    s = s + "\033[4m" + str(client) + "\033[0m" 
                else:
                    s = s +  str(client) 
    
                if(k != CLIENTS_PER_MODEL - 1):
                    s = s + ", "
                else:
                    s = s + "]"
            
            print("| CLIENT: ", i + 1, "\t| GROUP: ", j + 1, "\t= "  , s, "\t| TESTED at: ", ("\033[4m" + str(groupings[i][j][1]) + "\033[0m" ) if isMalicious(groupings[i][j][1]) and not SCENARIO else groupings[i][j][1] , "\t| ACCURACY: ", round(results[i][j],2), "\t|")

In [ ]:
rng = np.random.default_rng()

# intialize clients
clients_datasets = sample(getDataset(DATASET), IID, CLIENTS_COUNT,5)
# intialize model
global_model = Model()
# intialize global weights variable
global_weights = global_model.state_dict()
# set the model to use cpu or gpu
global_model.to('device')
# intialize metrics
metrics = []
epoch = 0
while (epoch < EPOCHS):
    print("epoch: ", epoch + 1)
    # intialize the dictionary of malicious clients
    dictionary = rng.choice(np.arange(CLIENTS_COUNT), size= int(CLIENTS_COUNT*ratio), replace=False)
    
    # intialize local wieghts list 
    local_weights = []
    
    # set the model to training mode
    global_model.train()
    
    for i in tqdm(range(CLIENTS_COUNT)):
        # train the ith client 
        weights = train(model=copy.deepcopy(global_model), global_round=epoch, client = i,  dataset = clients_datasets[i][0], 
            malicious = isMalicious(i) , scenario = SCENARIO)[0]
        # append the ith client's local weights to local_wieghts list
        local_weights.append(copy.deepcopy(weights))
    
    # grouping and mapping
    groupings = group(CLIENTS_COUNT, CLIENTS_PER_MODEL)
    
    # intialize the results 2D list
    results = np.empty([CLIENTS_COUNT, MODELS_COUNT])
    
    
    for i in tqdm(range(CLIENTS_COUNT)):
        for j in range(MODELS_COUNT):
    	
        	# aggregate weights of clients in the jth group of the ith client        
            weights = average(w = [local_weights[index] for index in groupings[i][j][0] ], bias = BIAS)
                
        	# intialize the model using the aggregated weights
            model = Model()
            model.load_state_dict(weights)
        	
        	# test the model at the respective client
            results[i][j] = test(model, clients_datasets[groupings[i][j][1]][1], 
                                     malicious = isMalicious(groupings[i][j][1]), 
                                     scenario = SCENARIO)[0]

    
    # prepare the results to test for reliability
    # intialize clients_outputs 2D list to keep track of all test result per testing client
    clients_outputs = []
    for i in range(CLIENTS_COUNT):
        clients_outputs.append([])
    
    for i in range(CLIENTS_COUNT):
        for j in range(MODELS_COUNT):
            (clients_outputs[groupings[i][j][1]]).append(results[i][j])
        
    
    # intialize clients_outputs list to keep track of mean test result per testing client
    mean_clients_outputs = [np.mean(client_outputs) for client_outputs in clients_outputs]
    
    # intialize list of unreliable clients
    unreliable = []
    # reliability metrics
    true_positives_unreliable = 0
    false_positives_unreliable = 0
    
    # check weather there are unreliable clients and we need to acknowledge_reliability
    acknowledge_reliability = isBimodal(mean_clients_outputs)
    
    if(acknowledge_reliability):
        # find splitting boundary using GMM
        reliability_boundary = findBoundary(mean_clients_outputs)
        above_reliability_boundary = 0
        below_reliability_boundary = 0
        
        # check weather the majority is above or below the reliability boundary    
        for i in range(CLIENTS_COUNT):
            if(mean_clients_outputs[i] <= reliability_boundary):
                below_reliability_boundary += 1
            else:
                above_reliability_boundary += 1
    
        for i in range(CLIENTS_COUNT):
            if(below_reliability_boundary > above_reliability_boundary):
                if(mean_clients_outputs[i] > reliability_boundary):
                    unreliable.append(i)
                    if(isMalicious(i)):
                        true_positives_unreliable += 1
                    else:
                        false_positives_unreliable += 1
            else:
                if(mean_clients_outputs[i] <= reliability_boundary):
                    unreliable.append(i)
                    if(isMalicious(i)):
                        true_positives_unreliable += 1
                    else:
                        false_positives_unreliable += 1
    
    penalties = np.zeros(CLIENTS_COUNT)
    for i in range(CLIENTS_COUNT):
        scores = []
        for j in range(MODELS_COUNT):
            if(isReliable(groupings[i][j][1])):
                scores.append(results[i][j])
        penalties[i] += np.mean(scores)

    poisoned = []
    true_positives_poisoned = 0
    false_positives_poisoned = 0

    score_boundary = findBoundary(penalties)
    poisoning_boundary = score_boundary
                
    # Count the number of clients above and below the poisoning boundary
    above_poisoning_boundary = sum(1 for penalty in penalties if penalty > poisoning_boundary)
    below_poisoning_boundary = len(penalties) - above_poisoning_boundary
            
    for i in range(CLIENTS_COUNT):
        if(below_poisoning_boundary > above_poisoning_boundary): 
            if(penalties[i] > poisoning_boundary):
                poisoned.append(i)
                if(isMalicious(i)):
                    true_positives_poisoned += 1
                else:
                    false_positives_poisoned += 1
        else:
            if(penalties[i] < poisoning_boundary):
                poisoned.append(i)
                if(isMalicious(i)):
                    true_positives_poisoned += 1
                else:
                    false_positives_poisoned += 1
    
    filtered_local_weights = []
    for i in range(CLIENTS_COUNT):
        if(not isPoisoned(i)):
            filtered_local_weights.append(local_weights[i])
    
    # aggregate local weights 
    global_weights = average(w = filtered_local_weights)
    
    # update global model
    global_model.load_state_dict(global_weights)
    
    
    # Calculate avg training accuracy over all users at every epoch
    accuracies, losses = [], []
    global_model.eval()
    for i in range(CLIENTS_COUNT):
        accuracy, loss = test(model=global_model, dataset = clients_datasets[i][1], malicious = False)
        accuracies.append(accuracy)
        losses.append(loss)
    
    metrics.append([true_positives_poisoned, false_positives_poisoned, np.mean(accuracies), np.mean(losses),true_positives_unreliable ,false_positives_unreliable ])
    epoch += 1

In [ ]:
x = np.arange(len(metrics)) + 1

In [ ]:
true_positives_poisoning = [x[0] for x in metrics]

In [ ]:
false_positives_poisoning = [x[1] for x in metrics]

In [ ]:
true_positives_reliability = [x[4] for x in metrics]

In [ ]:
false_positives_reliability = [x[5] for x in metrics]

In [ ]:
accuracy_y = [x[2] for x in metrics]
loss_y = [x[3] for x in metrics]

In [ ]:
plt.plot(x, true_positives_poisoning, "o-b", label="true positives poisoning")
plt.plot(x, false_positives_poisoning, "o:r", label = "false positives poisoning") 

plt.legend()
plt.title(DATASET,", "+ "S"+ str(SCENARIO) + ", "+  str(int(ratio*CLIENTS_COUNT)) + "/" + str(CLIENTS_COUNT))
plt.xlabel("epochs")
plt.ylabel("clients")
plt.show()  

In [ ]:
plt.plot(x, true_positives_reliability, "o-b", label="true positives reliability")
plt.plot(x, false_positives_reliability, "o:r", label = "false positives reliability") 

plt.legend()
plt.title(DATASET,", "+ "S"+ str(SCENARIO) + ", "+  str(int(ratio*CLIENTS_COUNT)) + "/" + str(CLIENTS_COUNT))
plt.xlabel("epochs")
plt.ylabel("clients")
plt.show()  

In [ ]:
array = np.add(true_positives_reliability , false_positives_reliability)

plt.plot(x, array, "o-r", label="flagged as unreliable")

plt.legend()
plt.title(DATASET,", "+ "S"+ str(SCENARIO) + ", "+  str(int(ratio*CLIENTS_COUNT)) + "/" + str(CLIENTS_COUNT))
plt.xlabel("epochs")
plt.ylabel("clients")
plt.show()  

In [ ]:
plt.plot(x, accuracy_y, "o-b")

plt.title(DATASET,", "+ "S"+ str(SCENARIO) + ", "+  str(int(ratio*CLIENTS_COUNT)) + "/" + str(CLIENTS_COUNT))
plt.xlabel("epochs")
plt.ylabel("mean accuracy")
plt.show()  

In [ ]:
plt.plot(x, loss_y, "o-r")

plt.title( DATASET,", "+ "S"+ str(SCENARIO) + ", "+  str(int(ratio*CLIENTS_COUNT)) + "/" + str(CLIENTS_COUNT))
plt.xlabel("epochs")
plt.ylabel("loss")
plt.show()  